# Which Method? — Interactive Comparison

This notebook demonstrates the different prediction methods in `online-cp` side by side on synthetic data.
See the [documentation guide](https://egonmedhatten.github.io/online-cp/guide/) for the full decision tree.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 4)

## 1. Regressor Comparison

We generate a nonlinear signal and compare Ridge, Kernel Ridge, and k-NN conformal regressors.

In [ ]:
# Generate nonlinear data
N = 200
X = np.sort(np.random.uniform(0, 2 * np.pi, (N, 1)), axis=0)
y = np.sin(X[:, 0]) + 0.2 * np.random.randn(N)

In [ ]:
from online_cp import (
    ConformalRidgeRegressor,
    KernelConformalRidgeRegressor,
    ConformalNearestNeighboursRegressor,
    GaussianKernel,
    IntervalWidth,
    ErrorRate,
)
from online_cp.evaluate import progressive_val

models = {
    "Ridge": ConformalRidgeRegressor(a=1.0),
    "Kernel Ridge": KernelConformalRidgeRegressor(kernel=GaussianKernel(sigma=1.0), a=0.01),
    "k-NN (k=5)": ConformalNearestNeighboursRegressor(k=5),
}

results = {}
for name, model in models.items():
    metric = ErrorRate() + IntervalWidth()
    progressive_val(model, X, y, epsilon=0.1, metric=metric)
    results[name] = metric.get()
    print(f"{name:20s} | Error rate: {results[name]['ErrorRate']:.3f} | Width: {results[name]['IntervalWidth']:.3f}")

**Takeaway**: Ridge produces wide intervals on nonlinear data (it can't model the curve). Kernel Ridge and k-NN adapt to the nonlinearity and produce tighter intervals while maintaining coverage.

## 2. CPS vs Regressor

A CPS gives you the full predictive distribution. Let's compare what you get from each.

In [ ]:
from online_cp import RidgePredictionMachine

# Train both on same data
X_train, y_train = X[:100], y[:100]
x_test = X[100:101]

# Regressor: gives an interval
reg = ConformalRidgeRegressor(a=1.0)
reg.learn_initial_training_set(X_train, y_train)
interval = reg.predict(x_test[0], epsilon=0.1)
print(f"Regressor interval (eps=0.1): ({interval.lower:.3f}, {interval.upper:.3f})")

# CPS: gives a full distribution
cps = RidgePredictionMachine(a=1.0)
cps.learn_initial_training_set(X_train, y_train)
cpd = cps.predict_cpd(x_test[0])

# Extract intervals at multiple levels from the CPD
for eps in [0.01, 0.05, 0.1, 0.2]:
    iv = cpd.predict_set(tau=0.5, epsilon=eps)
    print(f"  CPD interval (eps={eps}): ({iv.lower:.3f}, {iv.upper:.3f})")

**Takeaway**: The CPS gives you flexibility — one prediction, any significance level. The regressor is simpler if you only need one fixed $\varepsilon$.

## 3. Venn-Abers Calibrated Probabilities

Venn predictors give calibrated probabilities rather than sets.

In [ ]:
from online_cp import VennAbersPredictor
from online_cp.evaluate import progressive_val_venn
from online_cp.metrics import BrierScore, LogLoss, Width

# Binary classification data
N = 300
X_cls = np.random.randn(N, 3)
y_cls = (X_cls[:, 0] + X_cls[:, 1] + 0.5 * np.random.randn(N) > 0).astype(int)

vap = VennAbersPredictor(scorer="ridge", a=1.0)
metric = BrierScore() + LogLoss() + Width()
progressive_val_venn(vap, X_cls, y_cls, metric=metric)

print(f"Brier Score: {metric.get()['BrierScore']:.4f}")
print(f"Log Loss:    {metric.get()['LogLoss']:.4f}")
print(f"Width (p1-p0): {metric.get()['Width']:.4f}  (lower = more precise)")

## 4. Martingale Change Detection Comparison

We compare Plugin (with Gaussian KDE) vs SimpleJumper on a stream with an abrupt change.

In [ ]:
from online_cp import PluginMartingale, SimpleJumper, GaussianKDE, VilleWrapper

# Generate p-values: uniform for 200 steps, then biased towards 0
p_null = np.random.uniform(0, 1, 200)
p_alt = np.random.beta(0.5, 2, 100)  # Biased towards 0 (non-exchangeable)
p_values = np.concatenate([p_null, p_alt])

# Plugin martingale
plugin = PluginMartingale(betting_strategy=GaussianKDE, min_sample_size=50)
# Simple Jumper
jumper = SimpleJumper(J=0.01)

for p in p_values:
    plugin.update(p)
    jumper.update(p)

fig, ax = plt.subplots()
ax.semilogy(plugin.martingale_values, label="Plugin (GaussianKDE)")
ax.semilogy(jumper.martingale_values, label="SimpleJumper")
ax.axvline(200, color='red', linestyle='--', alpha=0.7, label="Change point")
ax.axhline(20, color='gray', linestyle=':', alpha=0.7, label="Threshold (20)")
ax.set_xlabel("Step")
ax.set_ylabel("Martingale value")
ax.set_title("Change Detection: Plugin vs Jumper")
ax.legend()
plt.tight_layout()
plt.show()

**Takeaway**: Both detect the change, but they may differ in detection delay and power depending on the type of shift. The Plugin adapts its betting density to the observed p-values; the Jumper uses a fixed grid of epsilon-experts.

## 5. Mondrian CP — Group-Conditional Coverage

Standard CP guarantees marginal coverage. Mondrian CP guarantees coverage **within each group**.

In [ ]:
from online_cp import ConformalRidgeRegressor
from online_cp.mondrian import MondrianConformalRegressor
from online_cp.evaluate import progressive_val
from online_cp.metrics import ErrorRate

# Two groups with different noise levels
N = 400
X_m = np.random.randn(N, 2)
group = (X_m[:, 0] > 0).astype(int)  # 0 or 1
noise = np.where(group == 0, 0.1, 1.0)  # Group 1 has 10x more noise
y_m = X_m[:, 1] + noise * np.random.randn(N)

# Use a warm-up set so the models are initialised before progressive_val
n_init = 20
X_init, y_init = X_m[:n_init], y_m[:n_init]
X_test, y_test = X_m[n_init:], y_m[n_init:]

# Standard CP
std_model = ConformalRidgeRegressor(a=1.0)
std_model.learn_initial_training_set(X_init, y_init)
std_metric = ErrorRate()
progressive_val(std_model, X_test, y_test, epsilon=0.1, metric=std_metric)

# Mondrian CP
mon_model = MondrianConformalRegressor(
    base_model=ConformalRidgeRegressor(a=1.0),
    category_fn=lambda x: int(x[0] > 0)
)
mon_model.learn_initial_training_set(X_init, y_init)
mon_metric = ErrorRate()
progressive_val(mon_model, X_test, y_test, epsilon=0.1, metric=mon_metric)

print(f"Standard CP error rate: {std_metric.get():.3f}")
print(f"Mondrian CP error rate: {mon_metric.get():.3f}")
print("\n(Both should be ~0.1, but Mondrian guarantees it PER GROUP)")

**Takeaway**: Standard CP may achieve 10% error overall but concentrate errors in one group. Mondrian CP ensures 10% error *in each group separately*, at the cost of wider intervals for the noisier group.